# Superhero Intelligence System
## Section 2 · Data Preparation (10 marks)

**Team:** Muhammad Wasay Tariq (24I-3071) · Syed Muhammad Rayyan Hassan

Both files are cleaned with the same fixed rules, in this order:

| # | Step | Applies to |
|---|---|---|
| 1 | **Inspect** — shape, columns, dtypes, missing counts (T1) | train.csv and test.csv |
| 2 | **Remove duplicates** — same name twice → keep first; name in both files → drop from train.csv | both |
| 3 | **Mark missing** — 6 scores numeric; blanks and 0 → NaN (0 means *unknown*) | both |
| 4 | **Remove weak rows** — 3 or more of the 6 scores missing → drop | both |
| 5 | **Fill the gaps** — training median, inside each task's Pipeline | training data only |

The cleaning code lives in `src/prep.py` so every task uses exactly the same rules.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":   # run from the repo root so data/ paths work
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from sklearn.impute import SimpleImputer

from src.prep import FEATURES, load_raw, clean, xy

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

### Step 1 · Inspect
The two files are read separately with `pd.read_csv("data/train.csv")` and `pd.read_csv("data/test.csv")`. They are never merged and we do not make our own train/test split.

In [2]:
train_raw, test_raw = load_raw()
data = clean(train_raw, test_raw)
data["shapes"]

,rows,columns
file,,
train.csv,1158,83
test.csv,290,83


**Table T1 · Columns, data types and missing counts (both files)**

In [3]:
data["t1"]

,dtype (train),missing (train),dtype (test),missing (test)
column,,,,
name,object,0,object,0
real_name,object,113,object,35
full_name,object,384,object,110
overall_score,object,0,object,0
history_text,object,72,object,17
powers_text,object,291,object,75
intelligence_score,int64,0,int64,0
strength_score,int64,0,int64,0
speed_score,int64,0,int64,0


*T1:* train.csv has 1158 rows and test.csv 290, both with the same 83 columns. None of the six score columns has a blank cell; the target `alignment` is blank for 66 train and 14 test rows, and all other missing values sit in columns we never use (text, physical traits, `has_…` flags).

**Table T1 (continued) · 0s and blanks turned into NaN in each score column (step 3)**

In [4]:
data["nan_counts"]

train.csv                                    test.csv                                   
                    0 -> NaN blank -> NaN text -> NaN total NaN 0 -> NaN blank -> NaN text -> NaN total NaN
score                                                                                                      
intelligence_score        85            0           0        85       20            0           0        20
strength_score            91            0           0        91       22            0           0        22
speed_score               86            0           0        86       20            0           0        20
durability_score          87            0           0        87       20            0           0        20
power_score               57            0           0        57       14            0           0        14
combat_score              85            0           0        85       20            0           0        20

*T1 (continued):* every unknown score is stored as a 0, never as a blank — 57–91 zeros per column in train.csv and 14–22 in test.csv. Step 3 turns all of them into NaN, so a 0 is treated as "unknown" rather than "no ability".

### Steps 2–4 · Duplicates, missing values, weak rows
Names are matched exactly, so different versions such as "Captain X" and "Captain X (Film)" are kept as separate characters.

**Table T2 · Rows left after every step**

In [5]:
data["t2"]

,train.csv,test.csv
step,,
Loaded,1158,290
After step 2 (duplicates),1158,290
After step 3 (mark missing),1158,290
After step 4 (weak rows),1072,270
Clean Set A,1072,not used
Clean Set B,916,225


*T2:* no name appears twice in a file or in both files, so step 2 removes nothing. Step 4 removes 86 train and 20 test characters with 3+ unknown scores (their zeros come in whole blocks), leaving Clean Set A = 1072 training characters and Clean Set B = 916 train / 225 test Good-or-Bad characters.

### Clean Set A and Clean Set B
* **Clean Set A → Task 2:** cleaned train.csv, every character (Good, Bad, Neutral or blank). Alignment is put aside and only looked at after clustering.
* **Clean Set B → Tasks 1 & 3:** cleaned train.csv and test.csv, only Good/Bad rows. Target: Good → 0 (Hero), Bad → 1 (Villain).

**Rows removed to build Clean Set B**

In [6]:
data["set_b_removed"]

,train.csv,test.csv
Neutral removed,93,31
blank removed,63,14


In [7]:
set_a, set_b_train, set_b_test = data["set_a"], data["set_b_train"], data["set_b_test"]

print("Clean Set A alignment mix (kept aside for Task 2):")
print(set_a["alignment"].fillna("blank").value_counts().to_string(), "\n")

classes = pd.DataFrame({
    "train.csv": set_b_train["y"].value_counts().sort_index(),
    "test.csv": set_b_test["y"].value_counts().sort_index(),
}).rename(index={0: "Good (Hero, 0)", 1: "Bad (Villain, 1)"})
classes

Clean Set A alignment mix (kept aside for Task 2):
alignment
Good       588
Bad        328
Neutral     93
blank       63 



,train.csv,test.csv
y,,
"Good (Hero, 0)",588,137
"Bad (Villain, 1)",328,88


*Set B:* building Set B drops 93 Neutral + 63 blank rows from train.csv and 31 Neutral + 14 blank rows from test.csv. The classes are imbalanced — 588 heroes vs 328 villains in train (35.8 % villains) and 137 vs 88 in test (39.1 %) — which is why Macro-F1 is the main metric in Tasks 1 and 3.

### Step 5 · Fill the gaps (inside each Pipeline)
The clean sets still contain NaNs on purpose. Each task puts `SimpleImputer(strategy="median")` first in its scikit-learn Pipeline, so the medians are learned from training data only and test.csv is only ever *transformed*. Below: how many rows still have a gap, and the medians the Task 1 imputer would learn from Clean Set B (train).

In [8]:
gaps = pd.DataFrame({
    name: df[FEATURES].isna().sum(axis=1).value_counts().sort_index()
    for name, df in [("Set A", set_a), ("Set B train", set_b_train), ("Set B test", set_b_test)]
}).fillna(0).astype(int).rename_axis("missing scores per row")
display(gaps)

X_train, y_train = xy(set_b_train)
imputer = SimpleImputer(strategy="median").fit(X_train)     # fit on TRAIN only - never on test.csv
pd.Series(imputer.statistics_, index=FEATURES, name="training median").to_frame().T

,Set A,Set B train,Set B test
missing scores per row,,,
0,1064,910,223
1,7,5,2
2,1,1,0


,intelligence_score,strength_score,speed_score,durability_score,power_score,combat_score
training median,85.0,30.0,45.0,60.0,75.0,80.0


After step 4 almost every character is complete: only 8 rows in Set A (7 with one gap, 1 with two) and 6 / 2 rows in Set B train / test still need a median fill, so the imputer changes very little.

### Sanity checks

In [9]:
assert not set(set_a["name"]) & set(test_raw["name"]), "a test name leaked into train"
for df in (set_a, set_b_train, set_b_test):
    assert (df[FEATURES].isna().sum(axis=1) < 3).all()            # step 4 held
    assert not (df[FEATURES] == 0).any().any()                     # no zeros left
    assert df["name"].is_unique
assert set(set_b_train["y"]) == set(set_b_test["y"]) == {0, 1}
assert len(set_a) == data["t2"].loc["Clean Set A", "train.csv"]
print("All checks passed.")

All checks passed.
